# Computer Vision

## Introduction

In [ ]:
import cv2
import sys
import numpy as np
import os

import matplotlib.pyplot as plt
from matplotlib import colormaps
from IPython.display import YouTubeVideo, display, HTML, Image, Video

## Image Processing
### Reading Images
### Example: Reading in the Starry Night image

In [ ]:
#| fig-align: center
#| fig-cap: "Starry Night original colours"
#| label: fig-starry-night-orig
#| fig-pos: 'ht'
#| echo: false

Image("data/starry_night.jpg", width=240)

In [ ]:
#| fig-align: center
#| fig-cap: "Starry Night reversed colour channels"
#| label: fig-starry-night-cv2
#| fig-pos: 'ht'

starry_night = cv2.imread('data/starry_night.jpg')
# Reversed (not correct)
plt.imshow(starry_night);

In [ ]:
#| eval: false

# plotting with cv2.imshow returns the correct colours.
# Hit any key on the keyboard to close window (do not hit the "X" button)

cv2.imshow('Starry Night', starry_night)
cv2.waitKey()
cv2.destroyWindow('Starry Night')

### Image Transformations
### Example: Common image transformations

In [ ]:
# create grayscale version of image
sn_grayscale = cv2.cvtColor(starry_night, cv2.COLOR_BGR2GRAY)

In [ ]:
rows,cols = starry_night.shape[:2]

# Create RGB version of translated image, moved 400 pixels to the right, and
# 100 pixels down. The translation matrix must be of the form:
# [ [1, 0, x-distance],
#   [0, 1, y-distance]]
M = np.float32([[1, 0, 400],[0, 1, 100]]) # Translation matrix
sn_translate = cv2.warpAffine(starry_night, M, (cols, rows))
sn_translate_rgb = cv2.cvtColor(sn_translate, cv2.COLOR_BGR2RGB)

In [ ]:
# Create RGB version of image, rotated 30 degrees anti-clockwise, about it's 
# centre.
rotation_matrix = cv2.getRotationMatrix2D((0.5*cols, 0.5*rows), 30, 1)
sn_rotated = cv2.warpAffine(starry_night, rotation_matrix, (cols, rows))
sn_rotated_rgb = cv2.cvtColor(sn_rotated, cv2.COLOR_BGR2RGB)

In [ ]:
# Resize the image by 20% in each direction, with cubic interpolation.
sn_resized = cv2.resize(starry_night, None,fx=1.2, fy=1.2, 
                        interpolation = cv2.INTER_CUBIC)
sn_resized_rgb = cv2.cvtColor(sn_resized, cv2.COLOR_BGR2RGB)
# Uncomment this line to save the new image and compare with the original.
# cv2.imwrite('data/test_sn.jpg', sn_resized)

In [ ]:
#| fig-align: center
#| fig-cap: "Image transformations"
#| label: fig-starry-night-transformations
#| fig-pos: 'ht'
#| echo: false

images = [sn_grayscale, sn_rotated_rgb, sn_translate_rgb, sn_resized_rgb]
titles = ['Grayscale', 'Rotated 30 degrees', 'Translated', 'Resized']

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, img, title in zip(axes.flat, images, titles):
    if title == 'Grayscale': 
        ax.imshow(img, cmap='gray') 
    else: 
        ax.imshow(img) 
    ax.set_title(title) 

plt.tight_layout() 

### Working with Masks

In [ ]:
#| fig-align: center
#| fig-cap: "Lionel Messi, footballer."
#| label: fig-messi-orig
#| fig-pos: 'ht'
#| echo: false

Image('data/messi5.jpg', width=480)

### Example: Creating a mask for the football

In [ ]:
# R=244, G=251, B=95
cv2.cvtColor(np.uint8([[[95, 251, 244 ]]]), cv2.COLOR_BGR2HSV)

In [ ]:
messi = cv2.imread('data/messi5.jpg')
hsv = cv2.cvtColor(messi, cv2.COLOR_BGR2HSV)

# define range of yellow color in HSV, using H=31, S=158, V=251 as the reference.
lower_yellow = np.array([21,  100, 100])
upper_yellow = np.array([41, 255, 255])

In [ ]:
#| fig-align: center
#| fig-cap: "Mask for football, and other yellow segments."
#| label: fig-messi-mask
#| fig-pos: 'ht'

# Threshold the HSV image to get only blue colors
mask1 = cv2.inRange(hsv, lower_yellow, upper_yellow)
plt.imshow(mask1, cmap='gray');

In [ ]:
mask1[:, :330] = 0
mask1[:, 405:] = 0

In [ ]:
#| fig-align: center
#| fig-cap: "Lionel Messi with outline of ball"
#| label: fig-messi-mask-2
#| fig-pos: 'ht'

#im2, contours= cv2.findContours(mask1, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

contours, _ = cv2.findContours(mask1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

largest_contour = max(contours, key=cv2.contourArea)
hull = cv2.convexHull(largest_contour)
cv2.drawContours(messi, [hull], -1, (0, 0, 255), thickness=2)

plt.imshow(cv2.cvtColor(messi, cv2.COLOR_BGR2RGB));

### Modifying Perspective of Images
### Example: Sudoku perspective transform

In [ ]:
#| fig-align: center
#| fig-cap: "Sudoku reference points"
#| label: fig-sudoku-orig
#| fig-pos: 'ht'

sudoku = cv2.imread('data/sudoku.png')

# Coordinates of original reference points
pts1 = np.float32([[73,85], [350, 88], [355, 360], [48, 357]])

# Create a polygon (clockwise) from reference points.
pts = np.array(pts1, np.int32)
img2 = cv2.polylines(sudoku, [pts], True, (255, 0, 0), 2 )
plt.imshow(img2);

In [ ]:
#| fig-align: center
#| fig-cap: "Sudoku original and transformed"
#| label: fig-sudoku-warped
#| fig-pos: 'ht'

# destination coordinates of original reference points.
pts2 = np.float32([[0,0],[300,0],[300,300],[0,300]])
M = cv2.getPerspectiveTransform(pts1,pts2)
sudoku_dst = cv2.warpPerspective(sudoku ,M,(300,300))
 
plt.subplot(121),plt.imshow(sudoku),plt.title('Input')
plt.subplot(122),plt.imshow(sudoku_dst),plt.title('Output');

In [ ]:
#| fig-align: center
#| fig-cap: "Football match screen capture"
#| label: fig-football-orig
#| fig-pos: 'ht'
#| echo: false

Image('data/football1.png', width=360)

### Example: Football screen capture

In [ ]:
football1 = cv2.imread('data/football1.png')
pts1 = np.float32([[45,121],[48,238],[206,230], [155,118]])
pts2 = np.float32([[45,100], [45,220], [77,220], [77,100]])

M = cv2.getPerspectiveTransform(pts1,pts2)
dst = cv2.warpPerspective(football1, M, (504, 360))

pts = np.array([[45,121],[48,238],[206,230], [155,119]], np.int32)
pts = pts.reshape((-1,1,2))
img2 = cv2.polylines(football1, [pts], True, (255, 0, 0), 1 )

In [ ]:
#| fig-align: center
#| fig-cap: "Football match transformed"
#| label: fig-football-warped
#| fig-pos: 'ht'
#| echo: false

fig = plt.figure(figsize=(10, 5)) 
gs = fig.add_gridspec(1, 2, width_ratios=[2, 1]) 

# First subplot
ax1 = fig.add_subplot(gs[0])
ax1.imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
ax1.set_title('Original Screen capture')

# Second subplot
ax2 = fig.add_subplot(gs[1])
ax2.imshow(cv2.cvtColor(dst[:300, :200], cv2.COLOR_BGR2RGB))
ax2.set_title('Transformed Image');

## Edge Detection
### Example: Sudoku edge detection

In [ ]:
dst_gray = cv2.cvtColor(sudoku_dst, cv2.COLOR_BGR2GRAY)
sobelx = cv2.Sobel(dst_gray, cv2.CV_64F, 1, 0,ksize=5)
sobely = cv2.Sobel(dst_gray, cv2.CV_64F, 0, 1,ksize=5)

In [ ]:
#| fig-align: center
#| fig-cap: "Sudoku vertical and horizontal edges"
#| label: fig-sudoku-edges
#| fig-pos: 'ht'

plt.subplot(1,2,1),plt.imshow(sobelx, cmap = 'gray')
plt.title('Sobelx: vertical edges'), plt.xticks([]), plt.yticks([])
plt.subplot(1,2,2),plt.imshow(sobely, cmap = 'gray')
plt.title('Sobely: horizontal edges'), plt.xticks([]), plt.yticks([]);

### Example: Canny edge detection of ball

In [ ]:
#| fig-align: center
#| fig-cap: "Lionel Messi, Canny edge"
#| label: fig-messi-canny
#| fig-pos: 'ht'

messi_gray = cv2.cvtColor(messi, cv2.COLOR_BGR2GRAY)
messi_canny_edge = cv2.Canny(messi_gray, 150, 200)
plt.imshow(messi_canny_edge, cmap='gray')

In [ ]:
blurred = cv2.GaussianBlur(messi_gray, (9, 9), 2)

circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1,
                            minDist=50,
                            param1=50,
                            param2=22,      # low threshold to start, 22 works best
                            minRadius=18,
                            maxRadius=38)

print(circles)

In [ ]:
#| fig-align: center
#| fig-cap: "Circles detected with Hough transform"
#| label: fig-messi-hough
#| fig-pos: 'ht'

circles_rounded = np.uint16(np.around(circles))
for i, c in enumerate(circles_rounded[0, :]): 
	cv2.circle(messi, (c[0], c[1]), c[2], (0, 255, 0), 2)     # circle 
	cv2.putText(messi, str(i), (c[0], c[1]),                  # label each one 
	            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

plt.imshow(cv2.cvtColor(messi, cv2.COLOR_BGR2RGB))
plt.title('Detected circles (numbered)');

## Computer Vision Demonstrations
### Models from OpenCV Repository {#sec-09-opencvmodels}

In [ ]:
#| eval: false
python .\download_models.py --save_dir GOOGLENET googlenet

In [ ]:
#| eval: false
# Googlenet from https://github.com/BVLC/caffe/tree/master/models/bvlc_googlenet
googlenet:
  load_info:
    url: "http://dl.caffe.berkeleyvision.org/bvlc_googlenet.caffemodel"
    sha1: "405fc5acd08a3bb12de8ee5e23a96bec22f08204"
  model: "bvlc_googlenet.caffemodel"
  config: "bvlc_googlenet.prototxt"
  mean: [104, 117, 123]
  scale: 1.0
  width: 224
  height: 224
  rgb: false
  classes: "classification_classes_ILSVRC2012.txt"
  sample: "classification"

In [ ]:
#| eval: false
python classification.py --model GOOGLENET/405fc5acd08a3bb12de8ee5e23a96bec22f08204/bvlc_googlenet.caffemodel \
--config /home/viknesh/NUS/coursesTaught/ind5003-book/opencv_extra/testdata/dnn/bvlc_googlenet.prototxt \
--width 224 --height 224 --classes ../data/dnn/classification_classes_ILSVRC2012.txt \
--mean 104 117 123 --input /home/viknesh/NUS/coursesTaught/ind5003-book/data/cars.jpg

### Models from OpenCV Model Zoo

In [ ]:
#| eval: false
python demo.py 

## Summary
## References
### Opencv documentation
### Books
### Github repositories
## Exercises

In [ ]:
#| fig-align: center
#| fig-cap: "Sudoku exercise"
#| label: fig-sudoku-exercise
#| fig-pos: 'ht'
#| echo: false

sudoku = cv2.imread('data/sudoku.png')
sudoku_gray = cv2.cvtColor(sudoku, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(sudoku_gray, threshold1=50, threshold2=150, apertureSize=3)

pts1 = np.float32([[70,82], [495, 85], [515, 515], [30, 515]])
pts = np.array(pts1, np.int32)
pts2 = np.float32([[0,0],[400,0],[400,400],[0,400]])
M = cv2.getPerspectiveTransform(pts1,pts2)
sudoku_straight = cv2.warpPerspective(edges,M, (400, 400))
plt.imshow(sudoku_straight, cmap='gray')